# EuroCropsML x OLMoEarth — Colab Benchmark (4 Models)

Compares **Nano**, **Tiny**, **Base**, and **Large** OLMoEarth models on EuroCropsML crop classification.

Everything downloads from HuggingFace — no local files needed.

**Pipeline:**
1. Clone repo + install deps
2. Download data from HF
3. Load dataset via streaming
4. Extract baseline features
5. Extract embeddings from 4 models
6. Classify + compare

---
## Section 0 — Environment Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/mahdikalantari555/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!pip install -r requirements.txt huggingface_hub umap-learn -q

In [ ]:
import sys, os, time, gc, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

import torch

from src.data.inspect import get_top_classes, generate_class_distribution
from src.data.streaming import stream_from_split
from src.data.features import combined_baseline_features
from src.encoder.olmoearth import OLMoEarthEncoder
from src.models.classical import get_classifier
from src.evaluate.metrics import compute_metrics, save_metrics
from src.viz.umap_viz import plot_umap
from src.viz.confusion import plot_confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import psutil
print(f"Python: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")
print(f"Device: {DEVICE}")

In [ ]:
# Paths
DATA_DIR = "./preprocess/preprocess"
SPLIT_DIR = "./split/split"
USE_CASE = "overlap_latvia_vs_estonia"
OUT_DIR = "results"

for d in ["dataset_stats", "features", "embeddings", "figures"]:
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)

---
## Section 1 — Download Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

REPO = "mahdi555/eurocrop-data"

for f in ["preprocess.zip", "split.zip"]:
    folder = f.replace(".zip", "")
    if os.path.exists(folder):
        print(f"Already extracted: {folder}/")
        continue
    print(f"Downloading {f} from HuggingFace...")
    path = hf_hub_download(repo_id=REPO, filename=f)
    print(f"Extracting...")
    with zipfile.ZipFile(path) as z:
        z.extractall(".")
    os.remove(path)

print("Done!")
!ls -d preprocess/ split/

---
## Section 2 — Load Dataset

In [ ]:
TOP_N = 20

top_classes = get_top_classes(DATA_DIR, TOP_N)
class_filter = set(top_classes)
label_encoder = LabelEncoder()
label_encoder.fit(top_classes)

print(f"Top-{TOP_N} classes: {top_classes[:5]} ...")

splits = {}
for split_key in ["train", "test"]:
    X_list, y_list = [], []
    for _, data, cls_label in stream_from_split(
        DATA_DIR, SPLIT_DIR, USE_CASE,
        split_name="all", split_key=split_key,
        class_filter=class_filter
    ):
        X_list.append(data)
        y_list.append(cls_label)
    if X_list:
        max_T = max(x.shape[0] for x in X_list)
        C = X_list[0].shape[1]
        X = np.zeros((len(X_list), max_T, C), dtype=np.float32)
        for i, x in enumerate(X_list):
            T = min(x.shape[0], max_T)
            X[i, :T, :] = x[:T, :]
        y = label_encoder.transform(y_list)
    else:
        X, y = np.array([]), np.array([])
    splits[split_key] = (X, y)
    del X_list, y_list
    gc.collect()

X_train, y_train = splits["train"]
X_test, y_test = splits["test"]
labels = sorted(set(y_test))

print(f"Train: {len(X_train)} | Test: {len(y_test)}")
print(f"Shape: {X_train.shape}")
print(f"Classes: {len(labels)}")

---
## Section 3 — Baseline Features

In [ ]:
print("Extracting baseline features...")
t0 = time.time()
feat_train = combined_baseline_features(X_train)
feat_test = combined_baseline_features(X_test)
print(f"Done in {time.time()-t0:.1f}s | Shape: {feat_train.shape}")

# Baseline: LightGBM on features
scaler = StandardScaler()
feat_tr_s = scaler.fit_transform(feat_train)
feat_te_s = scaler.transform(feat_test)
clf = get_classifier("lgbm", seed=SEED)
clf.fit(feat_tr_s, y_train)
y_pred_base = clf.predict(feat_te_s)
m_base = compute_metrics(y_test, y_pred_base, labels=labels)
m_base["method"] = "Baseline + LightGBM"
print(f"Baseline: OA={m_base['overall_accuracy']:.3f} F1={m_base['macro_f1']:.3f}")

---
## Section 4 — Multi-Model Embedding Extraction

Compare 4 OLMoEarth models: Nano, Tiny, Base, Large

In [ ]:
MODELS = {
    "Nano":  "allenai/OlmoEarth-v1_1-Nano",
    "Tiny":  "allenai/OlmoEarth-v1_1-Tiny",
    "Base":  "allenai/OlmoEarth-v1_1-Base",
    "Large": "allenai/OlmoEarth-v1_1-Large",
}

all_embeddings = {}
model_times = {}

for name, model_id in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Model: {name} ({model_id})")
    print(f"{'='*50}")

    t0 = time.time()
    encoder = OLMoEarthEncoder(
        mode="cloud",
        cloud_model_id=model_id,
        device=DEVICE
    )
    load_time = time.time() - t0
    print(f"  Loaded in {load_time:.1f}s")

    t0 = time.time()
    emb_train = encoder.encode(X_train, batch_size=32)
    emb_test = encoder.encode(X_test, batch_size=32)
    encode_time = time.time() - t0

    all_embeddings[name] = (emb_train, emb_test)
    model_times[name] = {"load": load_time, "encode": encode_time}

    print(f"  Train: {emb_train.shape} | Test: {emb_test.shape}")
    print(f"  Encoded in {encode_time:.1f}s")
    print(f"  Mean: {emb_train.mean():.4f} | Std: {emb_train.std():.4f}")

    np.save(f"{OUT_DIR}/embeddings/{name}_train.npy", emb_train)
    np.save(f"{OUT_DIR}/embeddings/{name}_test.npy", emb_test)

    del encoder, emb_train, emb_test
    gc.collect()

print(f"\nAll models encoded. Times: {model_times}")

---
## Section 5 — Classification per Model

In [ ]:
def run_experiment(name, X_tr, y_tr, X_te, y_te, clf_name, labels):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    clf = get_classifier(clf_name, seed=SEED)
    clf.fit(X_tr_s, y_tr)
    y_pred = clf.predict(X_te_s)
    m = compute_metrics(y_te, y_pred, labels=labels)
    m["method"] = name
    return m, y_pred

results = [m_base]
predictions = {"Baseline + LightGBM": y_pred_base}

for model_name, (emb_tr, emb_te) in all_embeddings.items():
    print(f"\n--- {model_name} ---")

    # Embeddings + LogReg
    m, yp = run_experiment(f"{model_name} + LogReg", emb_tr, y_train, emb_te, y_test, "logreg", labels)
    results.append(m)
    predictions[f"{model_name} + LogReg"] = yp
    print(f"  LogReg:  OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f}")

    # Embeddings + LightGBM
    m, yp = run_experiment(f"{model_name} + LightGBM", emb_tr, y_train, emb_te, y_test, "lgbm", labels)
    results.append(m)
    predictions[f"{model_name} + LightGBM"] = yp
    print(f"  LightGBM: OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f}")

    # Hybrid (features + embeddings) + LightGBM
    hybrid_tr = np.concatenate([feat_train, emb_tr], axis=1)
    hybrid_te = np.concatenate([feat_test, emb_te], axis=1)
    m, yp = run_experiment(f"{model_name} Hybrid + LightGBM", hybrid_tr, y_train, hybrid_te, y_test, "lgbm", labels)
    results.append(m)
    predictions[f"{model_name} Hybrid + LightGBM"] = yp
    print(f"  Hybrid:   OA={m['overall_accuracy']:.3f} F1={m['macro_f1']:.3f}")

---
## Section 6 — Results Comparison Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df[["method", "overall_accuracy", "macro_f1", "weighted_f1", "kappa"]].copy()
df.columns = ["Method", "Accuracy", "Macro F1", "Weighted F1", "Kappa"]
df = df.sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("=" * 80)
print("BENCHMARK RESULTS — 4 OLMoEarth Models")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

# Best per category
best = df.iloc[0]
print(f"\nBest overall: {best['Method']} (F1={best['Macro F1']:.3f})")

df.to_csv(f"{OUT_DIR}/colab_results.csv", index=False)
save_metrics(results, f"{OUT_DIR}/colab_all_results.json")
print(f"Saved to {OUT_DIR}/")

---
## Section 7 — Visualization

In [ ]:
# Bar chart comparing all methods
fig, ax = plt.subplots(figsize=(14, 6))
methods = df["Method"].values
f1s = df["Macro F1"].values
colors = ["steelblue" if "Baseline" in m else "coral" if "Hybrid" in m else "mediumseagreen" for m in methods]
ax.barh(range(len(methods)), f1s, color=colors)
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods, fontsize=9)
ax.set_xlabel("Macro F1")
ax.set_title("All Methods — Macro F1 Comparison")
ax.invert_yaxis()
for i, v in enumerate(f1s):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/all_methods_comparison.png", dpi=150)
plt.show()

In [ ]:
# UMAP for each model
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, (model_name, (emb_tr, _)) in enumerate(all_embeddings.items()):
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED)
    emb_2d = reducer.fit_transform(emb_tr[:500])
    y_sub = y_train[:500]
    unique_labels = np.unique(y_sub)
    for label in unique_labels:
        mask = y_sub == label
        axes[idx].scatter(emb_2d[mask, 0], emb_2d[mask, 1], s=5, alpha=0.7, label=f"{label}")
    axes[idx].set_title(f"{model_name}")
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])
plt.suptitle("UMAP — OLMoEarth Embeddings (Top-20 Classes)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/umap_all_models.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrix for best model
best_method = df.iloc[0]["Method"]
best_pred = predictions[best_method]

cm = __import__('sklearn.metrics', fromlist=['confusion_matrix']).confusion_matrix(y_test, best_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {best_method}')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/confusion_best.png", dpi=150)
plt.show()

In [ ]:
# Encoding time comparison
print("Encoding Time per Model:")
print(f"{'Model':<10} {'Load (s)':>10} {'Encode (s)':>12}")
print("-" * 35)
for name, t in model_times.items():
    print(f"{name:<10} {t['load']:>10.1f} {t['encode']:>12.1f}")

---
## Section 8 — Download Results

In [ ]:
from google.colab import files
import glob

print("Downloading results...")
for f in sorted(glob.glob(f"{OUT_DIR}/*.csv") + glob.glob(f"{OUT_DIR}/*.json")):
    print(f"  {f}")
    files.download(f)

print("\nDownloading figures...")
for f in sorted(glob.glob(f"{OUT_DIR}/figures/*.png")):
    print(f"  {f}")
    files.download(f)

print("\nAll results downloaded!")